In [1]:
# !pip install -q scikit-image

In [2]:
import os
import random
import re
import warnings
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import ResNet18_Weights, resnet18
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")


def detect_data_root():
    base = Path("/kaggle/input")
    if base.is_dir():
        subs = sorted([p for p in base.iterdir() if p.is_dir()])
        if subs:
            for p in subs:
                if re.search(r"anemia|ane|rbc|blood|prod", p.name, re.I):
                    return str(p)
            return str(subs[0])
    return str(Path("./data").resolve())


DATA_ROOT = "/kaggle/input/datasets/nehalshivane/anemiadataset2prod/AnemiaDataset2Prod"
print("DATA_ROOT =", DATA_ROOT)

# --- Dataset layout (AnemiaDataset2Prod) ---
# AneRBC-I/II -> Healthy_individuals / Anemic_individuals
CORPUS = "AneRBC-II"  # "AneRBC-I" or "AneRBC-II"
POOL_I_AND_II = False  # True: merge AneRBC-I + AneRBC-II (same class folder names)

CLASS_FOLDERS = ["Healthy_individuals", "Anemic_individuals"]
CLASS_NAMES = ["Healthy", "Anemic"]  # display only; labels 0,1 in folder order

SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 15
LR = 1e-4
NUM_WORKERS = 0
USE_REINHARD = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE =", DEVICE)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DATA_ROOT = /kaggle/input/datasets/nehalshivane/anemiadataset2prod/AnemiaDataset2Prod
DEVICE = cuda


In [3]:
from skimage.color import lab2rgb, rgb2lab


def reinhard_lab_uint8(source_rgb: np.ndarray, target_rgb: np.ndarray) -> np.ndarray:
    s = rgb2lab(source_rgb.astype(np.float64) / 255.0)
    t = rgb2lab(target_rgb.astype(np.float64) / 255.0)
    sm, tm = s.reshape(-1, 3).mean(0), t.reshape(-1, 3).mean(0)
    sstd = s.reshape(-1, 3).std(0) + 1e-6
    tstd = t.reshape(-1, 3).std(0) + 1e-6
    d = (s - sm) / sstd * tstd + tm
    rgb = lab2rgb(np.clip(d, 0, 100))
    return (np.clip(rgb, 0, 1) * 255.0).astype(np.uint8)

In [4]:
from skimage import filters, measure
from skimage.feature import graycomatrix, graycoprops


def _largest_cc_mask(gray: np.ndarray) -> np.ndarray:
    th = filters.threshold_otsu(gray)
    bw = gray > th
    lab = measure.label(bw)
    if lab.max() == 0:
        return np.ones_like(bw, dtype=bool)
    regions = measure.regionprops(lab)
    largest = max(regions, key=lambda r: r.area)
    return lab == largest.label


def extract_14d(rgb_uint8: np.ndarray) -> np.ndarray:
    gray = (
        0.299 * rgb_uint8[..., 0]
        + 0.587 * rgb_uint8[..., 1]
        + 0.114 * rgb_uint8[..., 2]
    ).astype(np.float64)
    g8 = gray.astype(np.uint8)
    mask = _largest_cc_mask(g8)
    if mask.sum() == 0:
        mask = np.ones_like(mask, dtype=bool)
    props = measure.regionprops(mask.astype(np.uint8), intensity_image=gray)[0]
    area = float(props.area)
    perim = float(props.perimeter)
    circ = (4.0 * np.pi * area) / (perim * perim + 1e-6)
    ecc = float(props.eccentricity)
    sol = float(props.solidity)
    ext = float(props.extent)

    gq = (gray / (gray.max() + 1e-6) * 255).astype(np.uint8)
    glcm = graycomatrix(
        gq, distances=[1], angles=[0], levels=256, symmetric=True, normed=True
    )
    contrast = float(graycoprops(glcm, "contrast")[0, 0])
    diss = float(graycoprops(glcm, "dissimilarity")[0, 0])
    homo = float(graycoprops(glcm, "homogeneity")[0, 0])
    energy = float(graycoprops(glcm, "energy")[0, 0])

    r = rgb_uint8[..., 0].astype(np.float64)
    g = rgb_uint8[..., 1].astype(np.float64)
    b = rgb_uint8[..., 2].astype(np.float64)
    mr = r[mask].mean() / 255.0
    mg = g[mask].mean() / 255.0
    mb = b[mask].mean() / 255.0
    ys, xs = np.nonzero(mask)
    cy, cx = ys.mean(), xs.mean()
    rr = np.sqrt((ys - cy) ** 2 + (xs - cx) ** 2)
    inn = rr < np.percentile(rr, 33)
    if inn.sum() < 5:
        pallor = 1.0
    else:
        inner = gray[mask][inn]
        outer = gray[mask][~inn]
        pallor = float(inner.mean() / (outer.mean() + 1e-6))

    return np.array(
        [area, perim, circ, ecc, sol, ext, contrast, diss, homo, energy, mr, mg, mb, pallor],
        dtype=np.float32,
    )

In [5]:
def list_images_for_class(class_dir: Path):
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
    out = []
    for p in class_dir.rglob("*"):
        if p.suffix.lower() in exts:
            out.append(str(p))
    return sorted(out)


root = Path(DATA_ROOT)
paths, labels = [], []

if POOL_I_AND_II:
    corpora = ["AneRBC-I", "AneRBC-II"]
else:
    corpora = [CORPUS]

for corp in corpora:
    base = root / corp
    if not base.is_dir():
        raise FileNotFoundError(f"Missing corpus folder: {base}")
    for li, folder_name in enumerate(CLASS_FOLDERS):
        cdir = base / folder_name
        if not cdir.is_dir():
            raise FileNotFoundError(f"Missing: {cdir}")
        ps = list_images_for_class(cdir)
        paths.extend(ps)
        labels.extend([li] * len(ps))
        print(corp, folder_name, len(ps))

print("Total images:", len(paths))

AneRBC-II Healthy_individuals 6000
AneRBC-II Anemic_individuals 6000
Total images: 12000


In [6]:
from pathlib import Path

def tree(p: Path, max_depth=4, depth=0):
    if depth > max_depth:
        return
    indent = "  " * depth
    print(f"{indent}{p.name}/")
    if p.is_dir() and depth < max_depth:
        try:
            for c in sorted(p.iterdir())[:50]:
                tree(c, max_depth, depth + 1)
        except Exception as e:
            print(f"{indent}  [err: {e}]")

root = Path("/kaggle/input/datasets/nehalshivane")
print("exists:", root.is_dir(), root)
if root.is_dir():
    tree(root, max_depth=4)

exists: True /kaggle/input/datasets/nehalshivane
nehalshivane/
  anemiadataset2prod/
    AnemiaDataset2Prod/
      AneRBC-I/
        Anemic_individuals/
        Healthy_individuals/
      AneRBC-II/
        Anemic_individuals/
        Healthy_individuals/


In [7]:
train_idx, val_idx = train_test_split(
    np.arange(len(paths)), test_size=0.2, random_state=SEED, stratify=labels
)
train_paths = [paths[i] for i in train_idx]
train_labels = [labels[i] for i in train_idx]
val_paths = [paths[i] for i in val_idx]
val_labels = [labels[i] for i in val_idx]
print("Train", len(train_paths), "Val", len(val_paths))

_ref_pil = Image.open(train_paths[0]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
REF_RGB = np.array(_ref_pil)

Train 9600 Val 2400


In [8]:
def raw_features_for_paths(path_list, use_reinhard: bool):
    X = []
    for p in tqdm(path_list, desc="Handcrafted (fit scaler)"):
        im = Image.open(p).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
        rgb = np.array(im)
        if use_reinhard:
            rgb = reinhard_lab_uint8(rgb, REF_RGB)
        X.append(extract_14d(rgb))
    return np.stack(X, axis=0)


X_train_raw = raw_features_for_paths(train_paths, USE_REINHARD)
scaler = StandardScaler()
scaler.fit(X_train_raw)

Handcrafted (fit scaler):   0%|          | 0/9600 [00:00<?, ?it/s]

StandardScaler()

In [9]:
imagenet_norm = transforms.Normalize(
    mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
)


class SmearDataset(Dataset):
    def __init__(self, path_list, label_list, scaler, train: bool, use_reinhard, ref_rgb):
        self.paths = path_list
        self.labels = label_list
        self.scaler = scaler
        self.train = train
        self.use_reinhard = use_reinhard
        self.ref_rgb = ref_rgb
        self.flip = transforms.RandomHorizontalFlip(p=0.5)
        self.vflip = transforms.RandomVerticalFlip(p=0.5)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        p = self.paths[i]
        y = self.labels[i]
        im = Image.open(p).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
        rgb = np.array(im)
        if self.use_reinhard:
            rgb = reinhard_lab_uint8(rgb, self.ref_rgb)
        xhand = extract_14d(rgb).astype(np.float64)
        xhand = self.scaler.transform(xhand.reshape(1, -1)).astype(np.float32).squeeze(0)
        t = transforms.ToTensor()(Image.fromarray(rgb))
        if self.train:
            t = self.flip(t)
            t = self.vflip(t)
        t = imagenet_norm(t)
        return t, torch.from_numpy(xhand), torch.tensor(y, dtype=torch.long)


train_ds = SmearDataset(train_paths, train_labels, scaler, True, USE_REINHARD, REF_RGB)
val_ds = SmearDataset(val_paths, val_labels, scaler, False, USE_REINHARD, REF_RGB)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

In [10]:
class HybridAttention(nn.Module):
    def __init__(self, num_classes: int = 2):
        super().__init__()
        w = ResNet18_Weights.IMAGENET1K_V1
        self.backbone = resnet18(weights=w)
        self.backbone.fc = nn.Identity()
        self.attn = nn.Sequential(nn.Linear(512 + 14, 256), nn.ReLU(), nn.Linear(256, 2))
        self.proj_cnn = nn.Linear(512, 256)
        self.proj_hand = nn.Linear(14, 256)
        self.head = nn.Sequential(nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, num_classes))

    def forward(self, x_img, x_hand):
        xc = self.backbone(x_img)
        z = torch.cat([xc, x_hand], dim=1)
        alpha = F.softmax(self.attn(z), dim=1)
        f = alpha[:, 0:1] * self.proj_cnn(xc) + alpha[:, 1:2] * self.proj_hand(x_hand)
        return self.head(f), alpha


class CNNBaseline(nn.Module):
    def __init__(self, num_classes: int = 2):
        super().__init__()
        w = ResNet18_Weights.IMAGENET1K_V1
        self.backbone = resnet18(weights=w)
        self.backbone.fc = nn.Identity()
        self.head = nn.Sequential(nn.ReLU(), nn.Dropout(0.3), nn.Linear(512, num_classes))

    def forward(self, x_img, x_hand=None):
        return self.head(self.backbone(x_img)), None


class HybridConcat(nn.Module):
    def __init__(self, num_classes: int = 2):
        super().__init__()
        w = ResNet18_Weights.IMAGENET1K_V1
        self.backbone = resnet18(weights=w)
        self.backbone.fc = nn.Identity()
        self.head = nn.Sequential(nn.ReLU(), nn.Dropout(0.3), nn.Linear(512 + 14, num_classes))

    def forward(self, x_img, x_hand):
        xc = self.backbone(x_img)
        z = torch.cat([xc, x_hand], dim=1)
        return self.head(z), None

In [11]:
@torch.no_grad()
def evaluate(model, loader, name="val"):
    model.eval()
    ys, ps, prob_pos = [], [], []
    alphas = []
    for xb, hb, yb in tqdm(loader, desc=name):
        xb, hb, yb = xb.to(DEVICE), hb.to(DEVICE), yb.to(DEVICE)
        logits, alpha = model(xb, hb)
        pred = logits.argmax(1)
        ys.append(yb.cpu().numpy())
        ps.append(pred.cpu().numpy())
        prob_pos.append(F.softmax(logits, dim=1)[:, 1].cpu().numpy())
        if alpha is not None:
            alphas.append(alpha.cpu().numpy())
    ys = np.concatenate(ys)
    ps = np.concatenate(ps)
    prob_pos = np.concatenate(prob_pos)
    acc = accuracy_score(ys, ps)
    prec, rec, f1, _ = precision_recall_fscore_support(
        ys, ps, average="macro", zero_division=0
    )
    try:
        auc = roc_auc_score(ys, prob_pos)
    except Exception:
        auc = float("nan")
    print(
        f"{name} Acc={acc:.4f} MacroP={prec:.4f} MacroR={rec:.4f} MacroF1={f1:.4f} AUC={auc}"
    )
    print(confusion_matrix(ys, ps))
    print(classification_report(ys, ps, target_names=CLASS_NAMES))
    if alphas:
        a = np.concatenate(alphas, 0)
        print("Alpha mean [CNN stream, hand stream]:", a.mean(0))
    return acc, prec, rec, f1


def train_model(model, name="model"):
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    crit = nn.CrossEntropyLoss()
    best_acc, best_sd = 0.0, None
    model.to(DEVICE)
    for ep in range(1, EPOCHS + 1):
        model.train()
        total, correct = 0, 0
        for xb, hb, yb in tqdm(train_loader, desc=f"{name} ep{ep}"):
            xb, hb, yb = xb.to(DEVICE), hb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            logits, _ = model(xb, hb)
            loss = crit(logits, yb)
            loss.backward()
            opt.step()
            total += yb.size(0)
            correct += (logits.argmax(1) == yb).sum().item()
        tr_acc = correct / max(total, 1)
        va, _, _, _ = evaluate(model, val_loader, name=f"{name} val ep{ep}")
        if va > best_acc:
            best_acc = va
            best_sd = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        print(f"Epoch {ep} train_acc={tr_acc:.4f} best_val={best_acc:.4f}")
    if best_sd is not None:
        model.load_state_dict(best_sd)
    return model, best_acc

In [12]:
print("=== CNN baseline ===")
cnn, cnn_best = train_model(CNNBaseline(), "cnn")
evaluate(cnn, val_loader, "cnn final")

=== CNN baseline ===


cnn ep1:   0%|          | 0/300 [00:00<?, ?it/s]

cnn val ep1:   0%|          | 0/75 [00:00<?, ?it/s]

cnn val ep1 Acc=0.8975 MacroP=0.8996 MacroR=0.8975 MacroF1=0.8974 AUC=0.9738326388888888
[[1121   79]
 [ 167 1033]]
              precision    recall  f1-score   support

     Healthy       0.87      0.93      0.90      1200
      Anemic       0.93      0.86      0.89      1200

    accuracy                           0.90      2400
   macro avg       0.90      0.90      0.90      2400
weighted avg       0.90      0.90      0.90      2400

Epoch 1 train_acc=0.8472 best_val=0.8975


cnn ep2:   0%|          | 0/300 [00:00<?, ?it/s]

cnn val ep2:   0%|          | 0/75 [00:00<?, ?it/s]

... (39 outputs left out)

cnn val ep15 Acc=0.9517 MacroP=0.9517 MacroR=0.9517 MacroF1=0.9517 AUC=0.9928173611111111
[[1135   65]
 [  51 1149]]
              precision    recall  f1-score   support

     Healthy       0.96      0.95      0.95      1200
      Anemic       0.95      0.96      0.95      1200

    accuracy                           0.95      2400
   macro avg       0.95      0.95      0.95      2400
weighted avg       0.95      0.95      0.95      2400

Epoch 15 train_acc=0.9637 best_val=0.9608


cnn final:   0%|          | 0/75 [00:00<?, ?it/s]

cnn final Acc=0.9608 MacroP=0.9618 MacroR=0.9608 MacroF1=0.9608 AUC=0.9952909722222223
[[1125   75]
 [  19 1181]]
              precision    recall  f1-score   support

     Healthy       0.98      0.94      0.96      1200
      Anemic       0.94      0.98      0.96      1200

    accuracy                           0.96      2400
   macro avg       0.96      0.96      0.96      2400
weighted avg       0.96      0.96      0.96      2400



(0.9608333333333333,
 0.9618391162977151,
 0.9608333333333333,
 0.9608119976431613)

In [13]:
print("=== Hybrid attention ===")
hyb, hyb_best = train_model(HybridAttention(), "hybrid_attn")
evaluate(hyb, val_loader, "hybrid final")

=== Hybrid attention ===


hybrid_attn ep1:   0%|          | 0/300 [00:00<?, ?it/s]

hybrid_attn val ep1:   0%|          | 0/75 [00:00<?, ?it/s]

hybrid_attn val ep1 Acc=0.9087 MacroP=0.9107 MacroR=0.9087 MacroF1=0.9086 AUC=0.9771902777777778
[[1132   68]
 [ 151 1049]]
              precision    recall  f1-score   support

     Healthy       0.88      0.94      0.91      1200
      Anemic       0.94      0.87      0.91      1200

    accuracy                           0.91      2400
   macro avg       0.91      0.91      0.91      2400
weighted avg       0.91      0.91      0.91      2400

Alpha mean [CNN stream, hand stream]: [0.8898206  0.11017863]
Epoch 1 train_acc=0.8551 best_val=0.9087


hybrid_attn ep2:   0%|          | 0/300 [00:00<?, ?it/s]

hybrid_attn val ep2:   0%|          | 0/75 [00:00<?, ?it/s]

... (39 outputs left out)

hybrid_attn val ep15 Acc=0.9413 MacroP=0.9413 MacroR=0.9413 MacroF1=0.9412 AUC=0.9899038194444445
[[1137   63]
 [  78 1122]]
              precision    recall  f1-score   support

     Healthy       0.94      0.95      0.94      1200
      Anemic       0.95      0.94      0.94      1200

    accuracy                           0.94      2400
   macro avg       0.94      0.94      0.94      2400
weighted avg       0.94      0.94      0.94      2400

Alpha mean [CNN stream, hand stream]: [0.96822184 0.03178085]
Epoch 15 train_acc=0.9626 best_val=0.9629


hybrid final:   0%|          | 0/75 [00:00<?, ?it/s]

hybrid final Acc=0.9629 MacroP=0.9633 MacroR=0.9629 MacroF1=0.9629 AUC=0.9956027777777777
[[1139   61]
 [  28 1172]]
              precision    recall  f1-score   support

     Healthy       0.98      0.95      0.96      1200
      Anemic       0.95      0.98      0.96      1200

    accuracy                           0.96      2400
   macro avg       0.96      0.96      0.96      2400
weighted avg       0.96      0.96      0.96      2400

Alpha mean [CNN stream, hand stream]: [0.97003853 0.02996335]


(0.9629166666666666,
 0.9632670123447524,
 0.9629166666666666,
 0.9629096542731777)